In [ ]:
import pandas as pd
import numpy as np  
from sodapy import Socrata
from datetime import datetime
import os
from tqdm import tqdm
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns


pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))
import utils

path = os.getcwd()
print(path)

path_repo = "C:/Users/PC/Desktop/tesis/"
path_data = "C:/Users/PC/Desktop/tesis/data/"

- La tabla es semanal, terminando en cada viernes

In [ ]:
qwe

## Tasas col V1

In [ ]:
tasascol = pd.read_csv(path_repo + "/data/2022.csv", decimal = ",", thousands = ".", encoding='utf-8')
tasascol.columns = [x.lower().replace(" ","_") for x in tasascol.columns]
tasascol.rename(columns = {"tasa_efectiva_promedio_ponderada":"tasa_ponderada", "montos_desembolsados":"monto", "numero_de_creditos_desembolsados":"num_creditos", 
                           "sum_numero_de_creditos":"num_creditos", "producto_de_crédito":"producto", "plazo_de_crédito":"plazo", "nombre_entidad":"banco"}, inplace=True)
tasascol.drop(columns = ["tipo_entidad", "codigo_entidad"], inplace =True)
tasascol.sample(5)

In [ ]:
cols_cat = ['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_persona', 'sexo', 'tamaño_de_empresa', 'tipo_de_crédito', 'tipo_de_garantía',
            'producto', 'plazo', 'grupo_etnico', 'antiguedad_de_la_empresa', 'tipo_de_tasa', 'clase_deudor', 'codigo_ciiu', 'codigo_municipio',
            'rango_monto_desembolsado']
cols_cont = ['tasa_ponderada', 'margen_adicional', 'monto', 'num_obl']

tasascol["fecha_corte"] = pd.to_datetime(tasascol["fecha_corte"], dayfirst=True, errors='coerce')

tasascol["tasa_ponderada"] = tasascol["tasa_ponderada"].astype(float)
tasascol["num_creditos"] = tasascol["num_creditos"].astype(int)
tasascol["fecha_corte"] = pd.to_datetime(tasascol["fecha_corte"])
tasascol["anomes"] = tasascol["fecha_corte"].dt.strftime("%Y-%m")

# tasascol = tasascol[~tasascol["banco"].isin(["Fondo Nacional del Ahorro", "La Hipotecaria", "Bancoldex", "Confiar",
#                                         "IRIS C.F.", "Bancamía S.A.", "Bancien", "Mibanco S.A.", "Banco Contactar",
#                                         'Tuya','Crezcamos','Credifamilia','Financiera Juriscoop C.F.','Rappipay',
#                                         'Banco GNB Sudameris','Banco Popular','Coopcentral','Banco Pichincha S.A.','Nu Colombia C.F.',
#                                         "Caja de Vivienda Militar y de Polícia"])] # Excluidos por tasas atipicos
                                        
# tasascol = tasascol[tasascol["tasa_ponderada"] <= 50]
# tasascol = tasascol[tasascol["num_creditos"] >= 2]
# tasascol = tasascol[(tasascol["rango_monto_desembolsado"].notna())] # tienen nulos antes de 202309

tasascol ["banco"] = tasascol["banco"].str.replace("Compañía De Financiamiento","CF").str.replace("Cooperativa Financiera","CF")
tasascol["plazo"] = tasascol["plazo"].replace({'Más de 1 año y hasta 3 años':"(1-3]", 'Más de 3 años y hasta 5 años':"(3-5]",
                                                'Más de 5 años y hasta 7 años':"(5-7]",
                                                'Más de 7 años y hasta 10 años':"(7-10]", 'Más de 10 años':"(10-]",
                                                "Hasta 6 meses":"(0-1]", "Más de 6 meses y hasta 1 año":"(0-1]",
                                                'Hasta 30 días':"(0-1]", 'Entre 31 días y 1 año':"(0-1]"})

tasascol["prom_monto_porobl"] = tasascol["monto"]/tasascol["num_creditos"]

print(tasascol["fecha_corte"].min(), "|", tasascol["fecha_corte"].max())

print(tasascol.shape)
print("Bancos:", tasascol["banco"].nunique())

tasascol.sample(7)

In [ ]:
tasascol[tasascol["anomes"] == "2022-08"]["fecha_corte"].value_counts()

In [ ]:
utils.vc(tasascol["tipo_de_crédito"])

In [ ]:
utils.vc(tasascol["tipo_de_garantía"])

In [ ]:
tasascol.isnull().mean()

In [ ]:
# Sample
tmp = tasascol.drop(columns = ['margen_adicional', 'grupo_etnico', 'antiguedad_de_la_empresa', 'clase_deudor', 'codigo_ciiu',
       'codigo_municipio', 'anomes', 'prom_monto_porobl']).sample(100)
tmp.to_csv(path_data+'sample.csv', index = False)
tmp.sample(5)

In [ ]:
print(np.percentile(tasascol["tasa_ponderada"], 0.05))
print(np.percentile(tasascol["tasa_ponderada"], 2.25))
print(np.percentile(tasascol["tasa_ponderada"], 5))
print(np.percentile(tasascol["tasa_ponderada"], 50))
print(np.percentile(tasascol["tasa_ponderada"], 95))
print(np.percentile(tasascol["tasa_ponderada"], 97.5))
print(np.percentile(tasascol["tasa_ponderada"], 99.5))

In [ ]:
print(tasascol[tasascol["tasa_ponderada"] <= 10].shape)
print(tasascol[tasascol["tasa_ponderada"] >= 40].shape)

In [ ]:
# ax = tasascol[tasascol["fecha_corte"] >= "2025-01-01"]
ax = tasascol
ax = ax.groupby("producto").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["producto", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
# ax = tasascol[tasascol["fecha_corte"] >= "2025-01-01"]
ax = tasascol
ax = ax.groupby("banco").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["banco", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
tasascol["anomes"].value_counts(dropna = False, normalize =True)
ax = tasascol
ax = ax.groupby(["anomes"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["mes", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
#ax.sort_values("monto", ascending=False).head(40)
ax

In [ ]:
ax = tasascol
ax = ax.groupby(["anomes", "plazo", "rango_desembolso"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":"mean"}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax["monto"] = ax["monto"]/1000000
ax["prom_monto_porobl"] = ax["monto"]/ax["num_creditos"] # millones

#ax.sort_values("monto", ascending=False).head(40)
tasascol = ax
tasascol

## How to group?

In [ ]:
print(tasascol.shape)
tasascolgr = tasascol.copy()
tasascolgr[cols_cat] = tasascolgr[cols_cat].fillna("NULL")
tasascolgr["tasaXmonto"]  = tasascolgr['tasa_ponderada']*tasascolgr['monto']
tasascolgr = tasascolgr.groupby(['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_crédito', 'tipo_de_garantía', 'tipo_de_persona', #'sexo', 'tamaño_de_empresa',  'producto',
                                'plazo', #'grupo_etnico', 'antiguedad_de_la_empresa','tipo_de_tasa', 
                                'rango_monto_desembolsado', #'clase_deudor',
                                #'codigo_ciiu', 'codigo_municipio', 'anomes', 'prom_monto_porobl'
                                ]).agg({"monto":"sum", "tasaXmonto":"sum", 'num_creditos':"sum"}).reset_index()
print(tasascolgr.shape)
tasascolgr.sample(5)

In [ ]:
601806204910.0000/30926219000.0000

In [ ]:
qwwe

## Tasas col V2

- Tabla semanal

In [64]:
tasascol = pd.read_csv(path_data+"tasascol_H2002-2026.csv", decimal = ",")
tasascol.columns = [x.lower().replace(" ","_") for x in tasascol.columns.tolist()]
tasascol.rename(columns={'año(aaaa)_-_semana(ss)':"semana", 'fecha_(dd/mm/aaaa)':"fecha", 'nombre_cuenta':"producto", 'clasificación_tipo_establecimiento':"tipo_entidad", 
                         'tasa_(%)':"tasa", 'monto_(miles_de_millones_cop)':"monto"}, inplace =True)
tasascol["fecha"] = pd.to_datetime(tasascol["fecha"], dayfirst = True)
tasascol["anomes"] = tasascol["fecha"].astype(str).str[:7]

print(len(tasascol))
tasascol

48961


,semana,fecha,producto,tipo_entidad,tasa,monto,anomes
0,2002-18,2002-05-03,Crédito de consumo,Bancos comerciales,28.2454,69367.5781,2002-05
1,2002-18,2002-05-03,Crédito de consumo,Compañías de financiamiento comercial,27.9476,10814.8918,2002-05
2,2002-18,2002-05-03,Crédito de consumo,Cooperativas financieras,27.6983,553.7210,2002-05
3,2002-18,2002-05-03,Crédito de consumo,Organismos cooperativos,29.1372,206.7000,2002-05
4,2002-18,2002-05-03,Crédito de consumo,Total establecimientos,28.2042,80942.8908,2002-05
...,...,...,...,...,...,...,...
48956,2026-42,2026-10-23,Tasa de interés de colocación sin Tesorería,Compañías de financiamiento comercial,20.6723,224703.2947,2026-10
48957,2026-42,2026-10-23,Tasa de interés de colocación sin Tesorería,Cooperativas financieras,20.7742,46398.0847,2026-10
48958,2026-42,2026-10-23,Tasa de interés de colocación sin Tesorería,Corporaciones financieras,16.0700,499.9418,2026-10
48959,2026-42,2026-10-23,Tasa de interés de colocación sin Tesorería,Entidades financieras especiales,17.1214,25851.3850,2026-10


In [ ]:
tasascol.tail(10)

In [ ]:
ax = tasascol[(tasascol["producto"] == "Tasa de interés de colocación Total") & (tasascol["tipo_entidad"] == "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
ax = ax.groupby(['anomes', 'producto', 'tipo_entidad'])[['tasaXMonto', 'monto']].sum().reset_index()
ax["tasa"] = ax["tasaXMonto"] / ax["monto"]
del ax["tasaXMonto"]

ax["vari_monto_mes"] = ax.groupby(["producto", "tipo_entidad"])["monto"].pct_change(1)*100
ax["vari_tasa_mes"] = ax.groupby(["producto", "tipo_entidad"])["tasa"].pct_change(1)*100
ax["monto_r12"] = ax.groupby(["producto", "tipo_entidad"])["monto"].shift(12)
ax["tasa_r12"] = ax.groupby(["producto", "tipo_entidad"])["tasa"].shift(12)
ax["y_crec_anual_monto"] = 100*((ax["monto"] / ax["monto_r12"]) - 1)
ax["crec_anual_tasa"] = 100*((ax["tasa"] / ax["tasa_r12"]) - 1)

ax.drop(columns = ["monto_r12", "tasa_r12"], inplace = True)
ax.tail(20)

In [ ]:
tasascol[(tasascol["producto"] == "Tasa de interés de colocación Total") & (tasascol["tipo_entidad"] == "Total establecimientos")].tail(4)

In [ ]:
# Valid ponderacion
display(tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Crédito de consumo") & (tasascol["tipo_entidad"] == "Total establecimientos")])

ax = tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Crédito de consumo")  & (tasascol["tipo_entidad"] != "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
print(ax["monto"].sum())
print((ax["tasaXMonto"].sum()/ax["monto"].sum()))
ax

In [ ]:
# Valid ponderacion
display(tasascol[(tasascol["fecha"] == "2026-10-23") & (tasascol["producto"] == "Tasa de interés de colocación Total")])# & (tasascol["tipo_entidad"] == "")])

ax = tasascol[(tasascol["fecha"] == "2026-10-23") & (~tasascol["producto"].isin(["Tasa de interés de colocación Total", "Tasa de interés de colocación Banco de la República Semanal",
                "Tasa de interés de colocación sin Tesorería"])) & (tasascol["tipo_entidad"] != "Total establecimientos")] #& (tasascol["producto"] == "Crédito de consumo")  & (tasascol["tipo_entidad"] != "Total establecimientos")]
ax["tasaXMonto"] = ax["tasa"] * ax["monto"]
ax = ax.groupby(["semana", "fecha", "tipo_entidad"])[["tasaXMonto", "monto"]].sum().reset_index()
ax["tasa"] = ax["tasaXMonto"] / ax["monto"]

print(ax["monto"].sum())
# print((ax["tasaXMonto"].sum()/ax["monto"].sum()))
ax

In [65]:
tasascol2 = tasascol.copy()

## Filters

In [66]:
tasascol = tasascol2.copy()
tasascol = tasascol[(tasascol["tipo_entidad"] == "Bancos comerciales")]
tasascol = tasascol[(~tasascol["producto"].isin(["Tasa de interés de colocación Total", "Tasa de interés de colocación Banco de la República Semanal",
                                                 "Tasa de interés de colocación sin Tesorería"]))]

tasascol["vari_monto_m1"] = tasascol.groupby(["producto", "tipo_entidad"])["monto"].pct_change(1)*100
tasascol["vari_tasa_m1"] = tasascol.groupby(["producto", "tipo_entidad"])["tasa"].pct_change(1)*100
tasascol["monto_r12"] = tasascol.groupby(["producto", "tipo_entidad"])["monto"].shift(12)
tasascol["tasa_r12"] = tasascol.groupby(["producto", "tipo_entidad"])["tasa"].shift(12)
tasascol["y_vari_anual_monto"] = 100*((tasascol["monto"] / tasascol["monto_r12"]) - 1)
tasascol["vari_anual_tasa"] = 100*((tasascol["tasa"] / tasascol["tasa_r12"]) - 1)

tasascol.drop(columns = ["monto_r12", "tasa_r12", "semana", "tipo_entidad"], inplace = True)

tasascol.tail(15)

,fecha,producto,tasa,monto,anomes,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa
48851,2026-10-09,Créditos Comerciales (Tesorería),13.9440,147685.8420,2026-10,-63.6048,9.3421,-42.9780,5.4579
48871,2026-10-16,Crédito de consumo,21.5475,1713192.1568,2026-10,-9.2091,0.1602,2.0597,-1.2283
48875,2026-10-16,Crédito popular productivo (total),55.6143,85714.4138,2026-10,-10.0226,-0.6280,7.1641,1.1105
48879,2026-10-16,Crédito productivo (total),34.4980,165748.8461,2026-10,-7.9750,1.9183,7.4305,5.0948
48883,2026-10-16,Crédito productivo de mayor monto (total),28.3976,101524.5854,2026-10,-5.6928,0.9672,-10.4794,1.9675
48887,2026-10-16,Créditos Comerciales (Ordinario),15.6460,2816541.1597,2026-10,-9.4058,-0.5290,1.6313,-1.0256
48893,2026-10-16,Créditos Comerciales (Preferencial o Corporativo),14.8717,1323397.1988,2026-10,-28.8000,-0.9219,-42.6833,0.1763
48896,2026-10-16,Créditos Comerciales (Tesorería),13.4884,224296.6923,2026-10,51.8742,-3.2674,90.4018,-1.9420
48916,2026-10-23,Crédito de consumo,21.1480,2183371.9504,2026-10,27.4447,-1.8543,-18.5712,0.1197
48920,2026-10-23,Crédito popular productivo (total),56.3362,104093.6270,2026-10,21.4424,1.2981,-8.1271,2.3186


In [ ]:
ax = tasascol[(tasascol["producto"].isin(["Créditos Comerciales (Ordinario)"]))].tail(200)
ax

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="y_vari_anual_monto", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
for i in tasascol["producto"].unique():
    ax = tasascol[tasascol["producto"] == i].tail(200)

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(15, 4))
    sns.lineplot(data=ax, x="fecha", y="monto", color="b", markersize = 5)  # y_vari_anual_monto
    plt.title(i, fontsize=14, fontweight="bold")
    plt.xlabel("", fontsize=12)
    plt.ylabel("", fontsize=12)
    plt.xticks(rotation=90, fontsize = 8)
    # plt.axhline(0, color = "gray", linestyle = "--")
    plt.tight_layout()
    plt.grid(None)
    plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="vari_monto_m1", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="monto", color="b", markersize = 5)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
# plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))
sns.lineplot(data=ax, x="fecha", y="tasa", color="b", markersize = 3)
plt.title("", fontsize=14, fontweight="bold")
plt.xlabel("", fontsize=12)
plt.ylabel("", fontsize=12)
plt.xticks(rotation=90, fontsize = 8)
# plt.axhline(0, color = "gray", linestyle = "--")
plt.tight_layout()
plt.grid(None)
plt.show()

In [ ]:
qwe

In [ ]:
# tasascol.tail(200).to_csv(path_data + "tasascol_sample.csv")

## BANREP

In [ ]:
def calc_viernes(df, columna_fecha='Fecha', saltar_fin_de_semana=False):
    df_resultado = df.copy()
    
    # Asegurar formato datetime
    fechas_dt = pd.to_datetime(df_resultado[columna_fecha])
    dia_semana = fechas_dt.dt.weekday
    
    # Calcular días de diferencia hasta el viernes (Viernes = 4)
    if saltar_fin_de_semana:
        dias_al_viernes = np.where(dia_semana > 4, 11 - dia_semana, 4 - dia_semana)
    else:
        dias_al_viernes = 4 - dia_semana
        
    # Asignar la nueva columna
    df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')
    
    return df_resultado


def filtrar_dias_habiles(df, columna_fecha='Fecha'):
    df_resultado = df.copy()
    
    # Asegurar que la columna tenga formato datetime
    fechas_dt = pd.to_datetime(df_resultado[columna_fecha])
    
    # .dt.weekday devuelve 0=Lunes hasta 6=Domingo. 
    # Los días menores a 5 son de lunes a viernes.
    mascara_habiles = fechas_dt.dt.weekday < 5
    
    # Retornar el DataFrame filtrado
    return df_resultado[mascara_habiles].copy()

In [ ]:
ph = [x for x in os.listdir(path_data+"BANREP/") if "trm" not in x]
print(len(ph))
ph

['agregados_crediticios.xlsx',
 'agregados_monetarios.xlsx',
 'indice_precios_vivienda.xlsx',
 'inflacion_consumidor.xlsx',
 'inflacion_consumidor2.xlsx',
 'ipp.xlsx',
 'ipp2.xlsx',
 'mercado_laboral.xlsx',
 'pib.xlsx',
 'pulso.xlsx',
 'reservas_internacionales.xlsx',
 'tasas_de_interes.xlsx',
 'tasas_de_interes2.xlsx',
 'tasas_de_interes3.xlsx',
 'unidad_valor_real.xlsx']

In [67]:
banrep = pd.read_excel(path_data+"BANREP/trm.xlsx", decimal = ",")
banrep = banrep.iloc[1:(len(banrep)-1)].dropna(subset = "Fecha")

for i in ph:
    tmp = pd.read_excel(path_data+"BANREP/"+i, decimal = ",")
    tmp = tmp.iloc[1:(len(tmp)-1)].dropna(subset = "Fecha")
    
    banrep["Fecha"] = banrep["Fecha"].astype(str)
    tmp["Fecha"] = tmp["Fecha"].astype(str)
    banrep = pd.merge(banrep, tmp, on = "Fecha", how = "outer")

    print(f"{i}: {tmp.shape}")

print("\n", banrep.shape)
# banrep1.columns = [x.lower().replace("í","i").replace("ó","o").replace("é","e").replace(" ", "_") for x in banrep1.columns]
banrep = banrep.replace({'-':np.nan})
banrep["Fecha"] = pd.to_datetime(banrep["Fecha"], dayfirst = True)
banrep["anomes"] = banrep["Fecha"].dt.strftime("%Y-%m")

banrep = banrep.sort_values("Fecha")

for i in [x for x in banrep if x not in ["Fecha", "anomes"]]:
    try:
        banrep[i] = banrep[i].astype(float)
    except:
        banrep[i] = banrep[i].str.replace(".","").str.replace(",",".").astype(float)

banrep = banrep[banrep["Fecha"] <= max(tasascol["fecha"])]

banrep.columns = [x.replace("Tasa de Certificado de Depósito a Término","Tasa CDT").replace("Tasa de Depósitos a Término Fijo","Tasa CDT").
                   replace("Tasa Representativa del Mercado (TRM)","TRM").replace("Indicador Bancario de Referencia (IBR)","IBR").
                   replace("Índice de precios al consumidor (IPC)","IPC").replace("Tasa de política monetaria","TPM").
                   replace("Captaciones a través de CDT","Tasa CDT") for x in banrep.columns]

dels = [x for x in banrep if "Variación porcentual" in x or "Dato fin de año" in x or "Dato fin de trimestre" in x or "Dato fin de semestre" in x]
banrep.drop(columns = dels, inplace = True)
print("After dels:", banrep.shape)

banrep = calc_viernes(banrep, columna_fecha='Fecha')

banrep = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if x not in ["Fecha", "Fecha_viernes"]]]

print("\n", banrep["Fecha"].min(), "|", banrep["Fecha"].max())
print(banrep.shape)
banrep.tail(8)

inflacion_consumidor.xlsx: (855, 17)
inflacion_consumidor2.xlsx: (867, 126)
tasas_de_interes.xlsx: (4897, 18)
tasas_de_interes2.xlsx: (9827, 23)
tasas_de_interes3.xlsx: (15689, 35)

 (16001, 236)
After dels: (15978, 105)

 1954-07-31 00:00:00 | 2026-10-23 00:00:00
(15978, 106)


C:\Users\PC\AppData\Local\Temp\ipykernel_15664\1419917051.py:15: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')


,Fecha,Fecha_viernes,TRM(Dato diario),TRM(Dato fin de semana),TRM(Dato fin de mes),TRM(Promedio mes corrido),TRM(Promedio mensual),TRM(Promedio anual),Euro - COP/EUR - Tasa media(Dato diario),Euro - COP/EUR - Tasa media(Dato fin de semana),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato diario),Libra esterlina - COP/GBP - Tasa media(Dato fin de semana),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),"Construcción de vivienda diferente de VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Construcción de vivienda VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Adquisición de vivienda diferente de VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Adquisición de vivienda VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Crédito de consumo, Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Ordinario), Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Preferencial o Corporativo), Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Tesorería), Tasa de interés, semanal(Dato semanal)","IBR a 1 mes, efectiva(Dato diario)","IBR a 1 mes, efectiva(Promedio semanal)","Crédito popular productivo (total), semanal(Dato semanal)","Crédito productivo (total), semanal(Dato semanal)","Crédito productivo de mayor monto(total), semanal(Dato semanal)","Tasa CDT por red de oficinas, diaria(Dato diario)","Tasa CDT por tesorería, diaria(Dato diario)","Tasa CDT por red de oficinas - Monto, diaria(Dato diario)","Tasa CDT por tesorería - Monto, diaria(Dato diario)",Prime(Dato diario),Prime(Promedio mes corrido),Prime(Promedio anual),"Microcréditos (diferentes de leasing), Tasa de interés, semanal(Dato semanal)","IBR overnight, efectiva(Dato diario)","IBR overnight, efectiva(Promedio semanal)","IBR a 3 meses, efectiva(Dato diario)","IBR a 3 meses, efectiva(Promedio semanal)",Operaciones interbancarias - Monto(Dato diario),"IBR a 6 meses, efectiva(Dato diario)","IBR a 6 meses, efectiva(Promedio semanal)","IBR a 12 meses, efectiva(Dato diario)","IBR a 12 meses, efectiva(Promedio semanal)","Tasa CDT (CDAT) entre 15 y 29 días, diaria(Dato diario)","Tasa CDT (CDAT) a 30 días, diaria(Dato diario)","Tasa CDT (CDAT) entre 31 y 90 días, diaria(Dato diario)","Tasa CDT (CDAT) entre 91 y 180 días, diaria(Dato diario)","Tasa CDT (CDT) a 120 días - Monto, diaria(Dato diario)_x","Tasa CDT (CDAT) entre 15 y 29 días - Monto, diaria(D

In [68]:
banrep_dia = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato diario" in x]]
print(banrep_dia.shape)

banrep_dia = calc_viernes(banrep_dia, columna_fecha='Fecha')
banrep_dia = filtrar_dias_habiles(banrep_dia, columna_fecha='Fecha')

print(banrep_dia.shape)
banrep_dia.tail(8)

(15978, 34)


C:\Users\PC\AppData\Local\Temp\ipykernel_15664\1419917051.py:15: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')


(11413, 34)


,Fecha,Fecha_viernes,TRM(Dato diario),Euro - COP/EUR - Tasa media(Dato diario),Libra esterlina - COP/GBP - Tasa media(Dato diario),"IBR a 1 mes, efectiva(Dato diario)","Tasa CDT por red de oficinas, diaria(Dato diario)","Tasa CDT por tesorería, diaria(Dato diario)","Tasa CDT por red de oficinas - Monto, diaria(Dato diario)","Tasa CDT por tesorería - Monto, diaria(Dato diario)",Prime(Dato diario),"IBR overnight, efectiva(Dato diario)","IBR a 3 meses, efectiva(Dato diario)",Operaciones interbancarias - Monto(Dato diario),"IBR a 6 meses, efectiva(Dato diario)","IBR a 12 meses, efectiva(Dato diario)","Tasa CDT (CDAT) entre 15 y 29 días, diaria(Dato diario)","Tasa CDT (CDAT) a 30 días, diaria(Dato diario)","Tasa CDT (CDAT) entre 31 y 90 días, diaria(Dato diario)","Tasa CDT (CDAT) entre 91 y 180 días, diaria(Dato diario)","Tasa CDT (CDT) a 120 días - Monto, diaria(Dato diario)_x","Tasa CDT (CDAT) entre 15 y 29 días - Monto, diaria(Dato diario)","Tasa CDT (CDAT) a 30 días - Monto, diaria(Dato diario)","Tasa CDT (CDAT) entre 31 y 90 días - Monto, diaria(Dato diario)","Tasa CDT (CDAT) entre 91 y 180 días - Monto, diaria(Dato diario)",TPM(Dato diario),"Tasa CDT (DTF) a 90 días, semanal(Dato diario)",Tasa interbancaria (TIB)(Dato diario),"Tasa CDT (CDT) a 360 días, diaria(Dato diario)","Tasa CDT (CDT) a 30 días, diaria(Dato diario)","Tasa CDT (CDT) a 120 días, diaria(Dato diario)","Tasa CDT (CDT) a 30 días - Monto, diaria(Dato diario)","Tasa CDT (CDT) a 120 días - Monto, diaria(Dato diario)_y","Tasa CDT (CDT) a 360 días - Monto, diaria(Dato diario)"
7112,2026-10-14,2026-10-16,3181.7400,3538.5721,4186.6926,12.2960,10.4400,12.8400,1627857.4400,638473.1900,7.0000,12.2640,12.4100,1110349.2900,12.5090,12.6790,1.4000,0.5800,1.9900,10.0400,72297.1100,41500.0000,930.0000,2904.0000,86785.0000,12.2500,10.3900,12.2400,12.6000,9.2300,10.6800,234548.4700,72297.1100,505870.9700
7626,2026-10-15,2026-10-16,3197.6500,3581.6878,4237.3659,12.3320,10.7000,12.9200,1613529.3500,398528.0600,7.0000,12.2600,12.3880,1207853.0800,12.4810,12.5990,NaN,0.6000,9.0400,10.1000,107435.0200,NaN,1284.0000,9945.0000,99649.0000,12.2500,10.3900,12.2500,12.4100,9.6500,11.1500,182729.3100,107435.0200,293766.2900
8141,2026-10-16,2026-10-16,3170.4600,3567.7186,4233.5152,12.2950,10.6700,12.8400,1893511.5600,432575.6900,7.0000,12.2570,12.3820,1391143.1900,12.4680,12.5630,NaN,0.3000,7.3600,10.0400,195130.3500,NaN,24259.0000,6294.0000,77952.0000,NaN,10.3900,12.2400,12.3100,9.6900,11.7200,189675.4600,195130.3500,193017.4500
9683,2026-10-19,2026-10-23,3174.6700,3561.0273,4227.8668,12.3080,10.4100,12.9600,2431194.5200,431695.8400,7.0000,12.2710,12.4210,1211591.8000,12.5110,12.6210,0.9500,NaN,2.0200,10.0600,241587.3200,20.0000,NaN,14217.0000,96378.0000,12.2500,10.4400,12.2500,12.3000,9.5000,11.2100,456467.6500,241587.3200,304277.0000
10197,2026-10-20,2026-10-23,3156.3200,3549.4396,4225.5234,12.3020,10.5900,13.1100,1251252.4800,639377.9200,7.0000,12.2450,12.4120,1378293.0300,12.4840,12.5870,1.4000,0.5000,0.8200,10.1900,58035.4300,6560.0000,496.0000,4627.0000,63207.0000,12.2500,10.4400,12.2000,12.6500,9.6300,10.4300,165705.5500,58035.4300,296417.5900
10711,2026-10-21,2026-10-23,3163.7600,3551.0042,4234.3764,12.3060,10.6400,12.9300,1781563.6200,981711.1500,7.0000,12.2590,12.4360,1279246.0100,12.5250,12.5980,NaN,0.3700,1.9000,10.1200,51873.8700,NaN,60.0000,3674.0000,73149.0000,12.2500,10.4400,12.2400,12.6800,9.6400,10.4200,186353.3100,51873.8700,578722.4800
11225,2026-10-22,2026-10-23,3191.4700,3573.6485,4265.2401,12.3090,10.8800,12.8600,1394934.5200,709293.6400,7.0000,12.2580,12.4380,1235634.6700,12.5170,12.6130,NaN,NaN,9.1800,10.1000,63704.1900,NaN,NaN,4780.0000,63663.0000,12.2500,10.4400,12.2500,12.7800,9.6900,10.4900,154674.9000,63704.1900,324297.1100
11739,2026-10-23,2026-10-23,3185.0100,3568.4852,4263.2951,12.3680,10.8700,12.9600,1508445.3600,573801.3600,7.0000,12.2670,12.4400,1396098.9800,12.5380,12.6070,0.7900,0.4000,4.6200,10.0100,61620.8600,4240.0000,240.0000,1695.0000,1005.9100,12

In [69]:
banrep_seman = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato semanal" in x]]
cols = [x for x in banrep_seman if x not in ["Fecha", "Fecha_viernes"]] 
banrep_seman = banrep_seman.dropna(subset=cols, how='all')

banrep_seman = calc_viernes(banrep_seman, columna_fecha='Fecha')
banrep_seman = filtrar_dias_habiles(banrep_seman, columna_fecha='Fecha')

print(banrep_seman.shape)
banrep_seman.tail(8)

(2232, 19)


C:\Users\PC\AppData\Local\Temp\ipykernel_15664\1419917051.py:15: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  df_resultado["Fecha_viernes"] = fechas_dt + pd.to_timedelta(dias_al_viernes, unit='d')


,Fecha,Fecha_viernes,"Construcción de vivienda diferente de VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Construcción de vivienda VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Adquisición de vivienda diferente de VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Adquisición de vivienda VIS (colocación en COP), Tasa de interés, semanal(Dato semanal)","Crédito de consumo, Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Ordinario), Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Preferencial o Corporativo), Tasa de interés, semanal(Dato semanal)","Créditos Comerciales (Tesorería), Tasa de interés, semanal(Dato semanal)","Crédito popular productivo (total), semanal(Dato semanal)","Crédito productivo (total), semanal(Dato semanal)","Crédito productivo de mayor monto(total), semanal(Dato semanal)","Microcréditos (diferentes de leasing), Tasa de interés, semanal(Dato semanal)",TPM(Dato semanal),"Tasa CDT (DTF) a 90 días, semanal(Dato semanal)","Tasa de interés de colocación Banco de la República, semanal(Dato semanal)","Tasa de interés de colocación Total, semanal(Dato semanal)","Tasa de interés de colocación sin Tesorería, semanal(Dato semanal)"
1926,2026-09-04,2026-09-04,15.2700,15.1100,15.3700,14.2900,21.3400,15.4600,13.7500,13.5100,54.0600,30.4200,26.5700,NaN,12.0000,10.1900,16.6000,16.5500,16.6100
5527,2026-09-11,2026-09-11,15.3000,15.4300,15.3300,14.3800,21.3000,14.8400,14.0100,13.5200,55.2100,31.0700,27.2200,NaN,12.0000,10.0500,16.3300,16.2800,16.3400
9126,2026-09-18,2026-09-18,15.2600,15.0700,15.3800,14.3400,21.4000,15.5800,14.6800,13.1500,55.7500,33.6800,28.1500,NaN,12.0000,10.2200,16.7700,16.7100,16.7800
12724,2026-09-25,2026-09-25,15.3700,15.1800,15.3800,14.4700,20.9600,15.2500,14.6300,13.0700,54.2200,32.8600,28.0200,NaN,12.0000,10.2600,16.4600,16.4000,16.4800
942,2026-10-02,2026-10-02,15.1800,15.1600,15.2700,14.1000,20.9500,15.1500,13.6800,12.7500,53.7400,32.8200,27.8200,NaN,12.2500,10.3800,16.2500,16.1400,16.2700
4542,2026-10-09,2026-10-09,15.5100,15.3600,15.3500,14.3800,21.5900,15.8300,15.0100,13.9400,56.3700,34.0900,28.2400,NaN,12.2500,10.1900,17.2400,17.1800,17.2500
8141,2026-10-16,2026-10-16,15.3300,15.2500,15.3300,14.4400,21.6600,15.8400,15.0100,13.4900,55.9900,34.7100,28.4800,NaN,NaN,10.3900,17.3200,17.2100,17.3400
11739,2026-10-23,2026-10-23,15.4100,15.2200,15.3600,14.3300,21.2500,14.9600,14.7500,13.8300,56.6500,35.2100,28.5100,NaN,12.2500,10.4400,16.4300,16.3700,16.4500


In [70]:
banrep_mes = banrep[["Fecha", "Fecha_viernes"] + [x for x in banrep if "Dato fin de mes" in x]]
cols = [x for x in banrep_mes if x not in ["Fecha", "Fecha_viernes"]] 
banrep_mes = banrep_mes.dropna(subset=cols, how='all')

print(banrep_mes.shape)
banrep_mes.tail(8)

(867, 39)


,Fecha,Fecha_viernes,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),TPM(Dato fin de mes),"Tasa CDT (DTF) a 90 días, semanal(Dato fin de mes)","Tasa CDT (DTF) a 90 días, mensual(Dato fin de mes)","Tasa de interés fin de mes de los certificados de depósito a término a 90 días, CDT 90 para bancos y corporaciones(Dato fin de mes)"
13987,2026-02-28,2026-02-27,3766.3000,4446.8704,5063.6020,5.2900,5.1600,5.5100,3.0200,6.4500,4.0500,2.6100,4.7300,7.8200,1.5100,1.6800,7.4400,9.6100,5.4500,5.8400,4.9300,155.7300,195.0000,143.9400,148.0700,163.2300,90.8700,149.6700,151.1900,153.1500,155.8300,159.9300,148.7700,172.2800,195.0000,10.2500,9.5900,9.5400,9.9300
15637,2026-03-31,2026-04-03,3669.9600,4228.5279,4839.5763,5.5600,5.3900,5.7900,3.0500,6.8000,4.1400,2.6100,5.2300,7.8700,4.7300,2.3800,7.5400,9.9200,6.0900,6.2700,5.1900,156.9400,197.4900,145.1600,149.4000,162.8800,93.5600,149.9300,152.6300,154.4300,157.0200,161.0400,149.7600,173.0300,197.4900,10.2500,9.9500,9.8300,9.7700
15062,2026-04-30,2026-05-01,3621.8600,4248.6229,4921.5645,5.6800,5.4400,5.7900,3.1300,6.7700,4.3400,2.6700,5.7900,8.2100,4.8800,2.5200,7.5800,9.6100,6.1100,6.7100,5.5100,158.1700,200.4700,146.1800,150.9900,164.2000,93.5100,150.0400,153.5900,155.6000,158.7300,162.7200,150.6700,174.3000,200.4700,11.2500,10.1400,10.1100,10.1600
15709,2026-05-31,2026-05-29,3678.1500,4292.2171,4957.9623,5.8400,5.7900,5.9800,3.1200,7.0400,5.1900,2.7000,6.1600,8.3500,5.3000,3.8400,7.5800,9.6200,6.0500,6.0400,5.6100,158.9100,200.4400,147.4400,151.7800,165.2000,93.7500,150.0400,153.9200,156.2800,159.6100,162.8200,151.5600,175.9100,200.4400,11.2500,10.0500,10.0300,9.5000
15177,2026-06-30,2026-07-03,3443.5900,3937.0564,4570.5048,6.1400,5.9800,6.0000,3.2300,7.0300,5.9100,2.9800,6.2200,8.3900,5.1600,3.9200,7.5700,9.5900,6.1200,6.8300,6.3700,159.5300,201.7700,148.2000,152.1700,165.2200,93.7300,150.0400,154.3800,156.8700,160.5300,163.4100,152.0400,176.5000,201.7700,11.2500,9.9000,9.9900,9.8100
15782,2026-07-31,2026-07-31,3132.4200,3604.0058,4215.4542,6.0300,6.0700,5.9600,3.2400,6.9600,6.4400,3.3800,6.3000,8.3700,4.9700,3.7400,7.5700,9.5300,5.9700,5.8200,6.1300,159.7900,201.5000,149.0100,152.5300,164.8300,93.7700,150.0400,154.2200,157.1000,160.6200,163.7500,152.4000,177.2200,201.5000,12.0000,10.2400,10.2000,10.0700
15855,2026-08-31,2026-09-04,3202.7900,3720.3609,4342.3427,6.2400,6.2700,6.1200,3.2900,7.1700,6.7700,3.2600,6.6700,8.3000,5.

In [ ]:
falta trimestral -> solo pib

In [ ]:
falta anual -> pib, salario minimo

## Captaciones

## Cartera vencida

In [71]:
venci = pd.read_csv(path_data + "cartera_vencida.csv", decimal = ",")
venci["FECHA_CORTE"] = pd.to_datetime(venci["FECHA_CORTE"], dayfirst = True)
print(venci.shape)
venci.tail(5)

C:\Users\PC\AppData\Local\Temp\ipykernel_15664\2538478098.py:1: DtypeWarning: Columns (0: (3) Vencida 1-2 Meses, 1: (4) Vencida 2-3 Meses, 2: (5) Vencida 1-3 Meses, 3: (6) Vencida 3-4 Meses, 4: (7) Vencida > de 4 Meses, 5: (9) Vencida +6 meses, 6: (10) Vencida 1-4 meses, 7: (11) Vencida 4-6 meses, 8: (12) Vencida 6-12 meses, 9: (13) Vencida 12-18 meses, 10: (14) Vencida > 12 meses, 11: (15) Vencida > 18 meses) have mixed types. Specify dtype option on import or set low_memory=False.
  venci = pd.read_csv(path_data + "cartera_vencida.csv", decimal = ",")


(112602, 34)


,TIPO_ENTIDAD,CODIGO_ENTIDAD,NOMBREENTIDAD,FECHA_CORTE,UNICAP,DESCRIP_UC,RENGLON,DESC_RENGLON,(1) Saldo de la cartera a la fecha de corte del reporte,(2) Vigente,(3) Vencida 1-2 Meses,(4) Vencida 2-3 Meses,(5) Vencida 1-3 Meses,(6) Vencida 3-4 Meses,(7) Vencida > de 4 Meses,(8) Vencida 3-6 Meses,(9) Vencida +6 meses,(10) Vencida 1-4 meses,(11) Vencida 4-6 meses,(12) Vencida 6-12 meses,(13) Vencida 12-18 meses,(14) Vencida > 12 meses,(15) Vencida > 18 meses,(16) Número de clientes Mora > 30 días,(17) Calificación de Riesgo A / Número de Clientes,(18) Calificación de Riesgo A / Saldo,(19) Calificación de Riesgo B / Número de Clientes,(20) Calificación de Riesgo B / Saldo,(21) Calificación de Riesgo C / Número de Clientes,(22) Calificación de Riesgo C / Saldo,(23) Calificación de Riesgo D / Número de Clientes,(24) Calificación de Riesgo D / Saldo,(25) Calificación de Riesgo E / Número de Clientes,(26) Calificación de Riesgo E / Saldo
112597,32,2,Jfk Cooperativa Financiera,2026-07-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,516.005.220,516.005.220,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,516.005.220,0,0,0,0,0,0,0,0
112598,32,4,Cotrafa Financiera,2026-08-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,9.208.278.866,9.208.278.866,0,0,0,0,0,0,0,0,0,0,0,0,0,0,105,9.208.278.866,0,0,0,0,0,0,0,0
112599,32,2,Jfk Cooperativa Financiera,2026-08-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,514.331.283,514.331.283,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,514.331.283,0,0,0,0,0,0,0,0
112600,32,5,Confiar Cooperativa Financiera,2026-08-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,5.868.437.692,5.868.437.692,0,0,0,0,0,0,0,0,0,0,0,0,0,0,40,5.868.437.692,0,0,0,0,0,0,0,0
112601,32,1,Cooperativa Financiera De Antioquia Cfa,2026-08-31,32,CREDITOS DE VIVIENDA PARA EMPL NO VIS,5,CRÉDITO EMPLEADOS VIVIENDA NO VIS,4.537.635.049,4.537.635.049,0,0,0,0,0,0,0,0,0,0,0,0,0,0,34,4.537.635.049,0,0,0,0,0,0,0,0


In [72]:
venci["FECHA_CORTE"].describe()

count                        112602
mean     2020-11-12 01:08:23.543454
min             2015-01-31 00:00:00
25%             2017-11-30 00:00:00
50%             2020-10-31 00:00:00
75%             2023-10-31 00:00:00
max             2026-08-31 00:00:00
Name: FECHA_CORTE, dtype: object

## Merge

In [ ]:
# Mensual
tasascol["anomes"] = tasascol["anomes"].astype(str)
banrep_mes["anomes"] = banrep_mes["Fecha"].astype(str).str[:7]

print(tasascol.shape)
df = pd.merge(tasascol, banrep_mes.drop(columns = ["Fecha", "Fecha_viernes"]), left_on = "anomes", right_on = "anomes", how = "left", indicator = True)
display(utils.vc(df["_merge"]))
print(df.shape)


# Semanal



# Diario



# Trimestal


# Anual





df = df.sort_values("fecha")

df.tail(5)

(6942, 9)


,_merge,count,porc
0,both,6914,99.5967
1,left_only,28,0.4033
2,right_only,0,0.0000


(6942, 47)


,fecha,producto,tasa,monto,anomes,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),TPM(Dato fin de mes),"Tasa CDT (DTF) a 90 días, semanal(Dato fin de mes)","Tasa CDT (DTF) a 90 días, mensual(Dato fin de mes)","Tasa de interés fin de mes de los certificados de depósito a término a 90 días, CDT 90 para bancos y corporaciones(Dato fin de mes)",_merge
6940,2026-10-23,Créditos Comerciales (Preferencial o Corporativo),14.7337,1671119.8933,2026-10,26.2750,-0.9281,-50.2798,1.5482,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
6936,2026-10-23,Crédito popular productivo (total),56.3362,104093.6270,2026-10,21.4424,1.2981,-8.1271,2.3186,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
6935,2026-10-23,Crédito de consumo,21.1480,2183371.9504,2026-10,27.4447,-1.8543,-18.5712,0.1197,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
6937,2026-10-23,Crédito productivo (total),35.0053,198558.4153,2026-10,19.7947,1.4704,-15.7119,7.0818,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
6941,2026-10-23,Créditos Comerciales (Tesorería),13.8267,290506.2712,2026-10,29.5187,2.5079,-9.8486,1.3678,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [79]:
df.iloc[:len(df)-28].tail(5)

,fecha,producto,tasa,monto,anomes,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),TPM(Dato fin de mes),"Tasa CDT (DTF) a 90 días, semanal(Dato fin de mes)","Tasa CDT (DTF) a 90 días, mensual(Dato fin de mes)","Tasa de interés fin de mes de los certificados de depósito a término a 90 días, CDT 90 para bancos y corporaciones(Dato fin de mes)",_merge
6909,2026-09-25,Crédito productivo (total),32.5500,194690.0918,2026-09,10.1020,-2.4741,37.6441,0.4073,3341.2300,3795.1361,4438.6570,6.2900,6.1800,6.0300,2.8700,7.1900,6.6900,3.1200,6.7500,7.7900,4.4500,4.9100,7.4900,9.4400,5.6300,6.7400,6.7100,161.0000,204.5100,150.0300,153.4900,165.0600,94.5100,152.2200,155.1000,158.2200,161.8300,165.3800,153.3000,177.9200,204.5100,12.0000,10.3800,10.2500,10.0100,both
6910,2026-09-25,Crédito productivo de mayor monto (total),27.9216,129616.5136,2026-09,7.0809,-0.6777,20.8287,0.7356,3341.2300,3795.1361,4438.6570,6.2900,6.1800,6.0300,2.8700,7.1900,6.6900,3.1200,6.7500,7.7900,4.4500,4.9100,7.4900,9.4400,5.6300,6.7400,6.7100,161.0000,204.5100,150.0300,153.4900,165.0600,94.5100,152.2200,155.1000,158.2200,161.8300,165.3800,153.3000,177.9200,204.5100,12.0000,10.3800,10.2500,10.0100,both
6911,2026-09-25,Créditos Comerciales (Ordinario),15.0948,4646315.6496,2026-09,2.0950,-2.4568,67.3639,-3.7060,3341.2300,3795.1361,4438.6570,6.2900,6.1800,6.0300,2.8700,7.1900,6.6900,3.1200,6.7500,7.7900,4.4500,4.9100,7.4900,9.4400,5.6300,6.7400,6.7100,161.0000,204.5100,150.0300,153.4900,165.0600,94.5100,152.2200,155.1000,158.2200,161.8300,165.3800,153.3000,177.9200,204.5100,12.0000,10.3800,10.2500,10.0100,both
6912,2026-09-25,Créditos Comerciales (Preferencial o Corporativo),14.4765,2292622.3850,2026-09,24.2761,-0.0069,70.8463,-0.6548,3341.2300,3795.1361,4438.6570,6.2900,6.1800,6.0300,2.8700,7.1900,6.6900,3.1200,6.7500,7.7900,4.4500,4.9100,7.4900,9.4400,5.6300,6.7400,6.7100,161.0000,204.5100,150.0300,153.4900,165.0600,94.5100,152.2200,155.1000,158.2200,161.8300,165.3800,153.3000,177.9200,204.5100,12.0000,10.3800,10.2500,10.0100,both
6913,2026-09-25,Créditos Comerciales (Tesorería),13.0664,223430.7597,2026-09,18.0882,-0.6382,-5.4444,-0.4433,3341.2300,3795.1361,4438.6570,6.2900,6.1800,6.0300,2.8700,7.1900,6.6900,3.1200,6.7500,7.7900,4.4500,4.9100,7.4900,9.4400,5.6300,6.7400,6.7100,161.0000,204.5100,150.0300,153.4900,165.0600,94.5100,152.2200,155.1000,1

In [80]:
df.sample(3)

,fecha,producto,tasa,monto,anomes,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),TPM(Dato fin de mes),"Tasa CDT (DTF) a 90 días, semanal(Dato fin de mes)","Tasa CDT (DTF) a 90 días, mensual(Dato fin de mes)","Tasa de interés fin de mes de los certificados de depósito a término a 90 días, CDT 90 para bancos y corporaciones(Dato fin de mes)",_merge
1729,2008-12-12,Créditos Comerciales (Ordinario),17.2569,1026177.5219,2008-12,-14.1482,0.0375,10.6196,0.2820,2243.5900,3118.7023,3225.7215,7.6700,5.3500,4.3800,2.3600,5.4300,9.2400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.4700,12.5400,69.8000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,69.4400,63.1700,71.2500,9.5000,9.8200,10.1200,9.7700,both
3657,2015-10-09,Microcréditos (Diferentes de Leasing),36.3819,82734.1687,2015-10,-35.1993,2.4696,-24.7572,0.3479,2897.8300,3201.0879,4475.4087,5.8900,4.8000,5.2000,6.4300,4.6000,3.5000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.4100,7.0400,86.9800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86.3800,82.7900,89.4400,4.7500,5.0800,4.7200,4.4800,both
4941,2020-05-22,Microcréditos (Diferentes de Leasing),31.3407,34294.2311,2020-05,18.0423,3.9452,-79.0539,-12.9938,3718.8200,4136.6294,4597.5772,2.8500,2.0000,2.0300,1.4200,2.2600,1.9000,0.3300,0.0400,4.4800,-4.8900,3.0700,5.9600,3.3700,2.5000,7.5700,4.5500,105.3600,112.2600,104.5300,102.0500,102.9300,96.8700,110.9000,104.6000,104.8800,106.1100,105.2800,104.1400,105.3800,112.2600,3.2500,4.2500,4.2900,4.0200,both


In [94]:
var = "monto"
cols = [x for x in df if x not in ["fecha", "producto", "anomes", "_merge"]]
ax = df[df["producto"] == "Créditos Comerciales (Ordinario)"][cols]
ax = ax.iloc[:len(ax)-28]
ax

,tasa,monto,vari_monto_m1,vari_tasa_m1,y_vari_anual_monto,vari_anual_tasa,TRM(Dato fin de mes),Euro - COP/EUR - Tasa media(Dato fin de mes),Libra esterlina - COP/GBP - Tasa media(Dato fin de mes),Inflación total anual(Dato fin de mes),Inflación sin alimentos(Dato fin de mes),Inflación sin alimentos ni regulados(Dato fin de mes),"Inflación de bienes, sin alimentos ni regulados(Dato fin de mes)","Inflación de servicios, sin alimentos ni regulados(Dato fin de mes)",Inflación de regulados(Dato fin de mes),"Inflación, Prendas de vestir y calzado(Dato fin de mes)","Inflación, Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)","Inflación, Salud(Dato fin de mes)","Inflación, Información y comunicación(Dato fin de mes)","Inflación, Recreación y cultura(Dato fin de mes)","Inflación, Educación(Dato fin de mes)","Inflación, Restaurantes y hoteles(Dato fin de mes)","Inflación, Bienes y servicios diversos(Dato fin de mes)",Inflación de alimentos(Dato fin de mes),Inflación de alimentos y regulados(Dato fin de mes),Índice de Precios al Consumidor (IPC)(Dato fin de mes),IPC: Alimentos y bebidas no alcohólicas(Dato fin de mes),"IPC: Alojamiento, agua, electricidad, gas y otros combustibles(Dato fin de mes)","IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",IPC: Transporte(Dato fin de mes),IPC: Información y comunicación(Dato fin de mes),IPC: Educación(Dato fin de mes),IPC: Bienes y servicios diversos(Dato fin de mes),"IPC: Bogotá, D.C.(Dato fin de mes)",IPC: Cali(Dato fin de mes),IPC: Bucaramanga(Dato fin de mes),IPC: Sin alimentos(Dato fin de mes),IPC: Regulados(Dato fin de mes),IPC: de alimentos(Dato fin de mes),TPM(Dato fin de mes),"Tasa CDT (DTF) a 90 días, semanal(Dato fin de mes)","Tasa CDT (DTF) a 90 días, mensual(Dato fin de mes)","Tasa de interés fin de mes de los certificados de depósito a término a 90 días, CDT 90 para bancos y corporaciones(Dato fin de mes)"
1,14.7691,213314.1142,NaN,NaN,NaN,NaN,2321.1600,NaN,NaN,5.8400,5.1400,4.6700,4.6200,4.6900,7.4700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1300,7.8900,48.6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.4500,38.0100,43.8100,5.7500,9.0100,9.0700,8.5000
6,14.5591,175259.1509,-17.8399,-1.4220,NaN,NaN,2321.1600,NaN,NaN,5.8400,5.1400,4.6700,4.6200,4.6900,7.4700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1300,7.8900,48.6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.4500,38.0100,43.8100,5.7500,9.0100,9.0700,8.5000
12,15.7095,183265.2631,4.5682,7.9017,NaN,NaN,2321.1600,NaN,NaN,5.8400,5.1400,4.6700,4.6200,4.6900,7.4700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1300,7.8900,48.6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.4500,38.0100,43.8100,5.7500,9.0100,9.0700,8.5000
17,14.4721,187392.4187,2.2520,-7.8767,NaN,NaN,2321.1600,NaN,NaN,5.8400,5.1400,4.6700,4.6200,4.6900,7.4700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1300,7.8900,48.6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.4500,38.0100,43.8100,5.7500,9.0100,9.0700,8.5000
22,14.9590,207307.0935,10.6273,3.3645,NaN,NaN,2321.1600,NaN,NaN,5.8400,5.1400,4.6700,4.6200,4.6900,7.4700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.1300,7.8900,48.6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.4500,38.0100,43.8100,5.7500,9.0100,9.0700,8.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6715,14.6865,2909014.7532,18.1647,0.3560,-54.9714,15.3314,3669.9600,4228.5279,4839.5763,5.5600,5.3900,5.7900,3.0500,6.8000,4.1400,2.6100,5.2300,7.8700,4.7300,2.3800,7.5400,9.9200,6.0900,6.2700,5.1900,156.9400,197.4900,145.1600,149.4000,162.8800,93.5600,149.9300,152.6300,154.4300,157.0200,161.0400,149.7600,173.0300,197.4900,10.2500,9.9500,9.8300,9.7700
6722,14.6063,3169385.5998,8.9505,-0.5462,-31.8246,16.4211,3669.9600,4228.5279,4839.5763,5.5600,5.3900,5.7900,3.0500,6.8000,4.1400,2.6100,5.2300,7.8700,4.7300,2.3800,7.5400,9.9200,6.0900,6.2700,5.1900,156.9400,197.4900,145.1600,149.4000,162.

In [95]:
_cor = ax.corr().reset_index()
_cor = pd.melt(_cor, id_vars = "index")
_cor = _cor[_cor["index"] != _cor["variable"]]
_cor = _cor[_cor["variable"] == var].sort_values("value", ascending = False).drop(columns = "variable")
_cor

,index,value
81,IPC: de alimentos(Dato fin de mes),0.8817
80,IPC: Regulados(Dato fin de mes),0.8792
68,Índice de Precios al Consumidor (IPC)(Dato fin de mes),0.8749
79,IPC: Sin alimentos(Dato fin de mes),0.8697
76,"IPC: Bogotá, D.C.(Dato fin de mes)",0.7935
78,IPC: Bucaramanga(Dato fin de mes),0.7934
77,IPC: Cali(Dato fin de mes),0.7888
71,"IPC: Muebles, artículos para el hogar y para la conservación ordinaria del hogar(Dato fin de mes)",0.7736
75,IPC: Bienes y servicios diversos(Dato fin de mes),0.7703
72,IPC: Transporte(Dato fin de mes),0.7656
